In [ ]:

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

# =========================
# Config
# =========================
SKIPATOM_CSV = "skipatom_20201009_induced.csv"   # element -> embedding table
INPUT_FILE   = "SMILES_original.csv"           
OUTPUT_FILE  = "polymer_skipatom_env_vectors.csv"

INCLUDE_HYDROGENS = False  # usually False for "environment"; True makes many sites differ mainly by H count
RADIUS = 2                 # neighbourhood radius: 1 or 2 are most common


# =========================
# IO helpers
# =========================
def load_smiles_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.lower().strip() for c in df.columns]

    # If columns are weird, try to coerce
    if "smiles" not in df.columns:
        df.columns = ["name", "smiles"] + list(df.columns[2:])
    if "name" not in df.columns:
        df["name"] = ""

    df = df.dropna(subset=["smiles"]).copy()
    df["smiles"] = df["smiles"].astype(str).str.strip()
    df["name"] = df["name"].astype(str).str.strip()
    return df


def load_skipatom_table(path: str):
    emb_df = pd.read_csv(path)
    emb_df.columns = [str(c).strip() for c in emb_df.columns]

    sym_col = emb_df.columns[0]
    vec_cols = list(emb_df.columns[1:])

    elem_to_vec = {}
    dims = set()

    for _, row in emb_df.iterrows():
        sym = str(row[sym_col]).strip()
        v = row[vec_cols].to_numpy(dtype=np.float32)
        elem_to_vec[sym] = v
        dims.add(v.shape[0])

    if len(dims) != 1:
        raise ValueError(f"SkipAtom table has inconsistent dims: {sorted(dims)}")

    dim = dims.pop()
    print(f"Loaded SkipAtom embeddings for {len(elem_to_vec)} elements. dim={dim}")
    return elem_to_vec, dim


# ==============================
# Graph neighbourhood utilities
# ==============================
def get_shell_indices(mol: Chem.Mol, center_idx: int, radius: int):
    """
    Returns dict shells[k] = set(atom indices at graph distance exactly k from center_idx)
    for k = 1..radius
    """
    visited = {center_idx}
    frontier = {center_idx}
    shells = {}

    for k in range(1, radius + 1):
        next_frontier = set()
        for idx in frontier:
            atom = mol.GetAtomWithIdx(idx)
            for nbr in atom.GetNeighbors():
                j = nbr.GetIdx()
                if j not in visited:
                    next_frontier.add(j)
        shells[k] = next_frontier
        visited |= next_frontier
        frontier = next_frontier

    return shells


def agg_stats(vecs: list[np.ndarray], dim: int):
    """
    Returns (mean, std) for a list of vectors.
    If empty, returns zeros.
    """
    if not vecs:
        return np.zeros((dim,), dtype=np.float32), np.zeros((dim,), dtype=np.float32)
    V = np.vstack(vecs).astype(np.float32)
    return V.mean(axis=0), V.std(axis=0)


# =========================
# Atom-level feature builders
# =========================
def atom_local_scalar_features(atom: Chem.Atom) -> np.ndarray:
    """
    Small set of RDKit-derived local descriptors.
    Kept compact and fixed-length.
    """
    hyb = atom.GetHybridization()
    # One-hot-ish hybridisation (common ones)
    hyb_sp   = 1.0 if hyb == Chem.rdchem.HybridizationType.SP else 0.0
    hyb_sp2  = 1.0 if hyb == Chem.rdchem.HybridizationType.SP2 else 0.0
    hyb_sp3  = 1.0 if hyb == Chem.rdchem.HybridizationType.SP3 else 0.0
    hyb_other = 1.0 if (hyb_sp + hyb_sp2 + hyb_sp3) == 0.0 else 0.0

    feats = np.array([
        float(atom.GetDegree()),
        float(atom.GetTotalDegree()),
        float(atom.GetExplicitValence()),
        float(atom.GetImplicitValence()),
        float(atom.GetFormalCharge()),
        float(atom.GetTotalNumHs()),
        1.0 if atom.GetIsAromatic() else 0.0,
        1.0 if atom.IsInRing() else 0.0,
        hyb_sp, hyb_sp2, hyb_sp3, hyb_other
    ], dtype=np.float32)
    return feats


def atom_environment_vector(
    mol: Chem.Mol,
    atom_idx: int,
    elem_to_vec: dict,
    dim: int,
    radius: int = 2
):
    """
    Builds a per-atom environment vector:
      [center_embed,
       shell1_mean, shell1_std,
       shell2_mean, shell2_std (if radius>=2),
       local_scalar_features]
    """
    atom = mol.GetAtomWithIdx(atom_idx)
    sym = atom.GetSymbol()

    center = elem_to_vec.get(sym)
    if center is None:
        return None, {"unknown_center": True}

    shells = get_shell_indices(mol, atom_idx, radius=radius)

    # 1-hop
    shell1_vecs = []
    unknown1 = 0
    for j in shells.get(1, set()):
        v = elem_to_vec.get(mol.GetAtomWithIdx(j).GetSymbol())
        if v is None:
            unknown1 += 1
        else:
            shell1_vecs.append(v)
    s1_mean, s1_std = agg_stats(shell1_vecs, dim)

    parts = [center, s1_mean, s1_std]

    unknown2 = 0
    if radius >= 2:
        shell2_vecs = []
        for j in shells.get(2, set()):
            v = elem_to_vec.get(mol.GetAtomWithIdx(j).GetSymbol())
            if v is None:
                unknown2 += 1
            else:
                shell2_vecs.append(v)
        s2_mean, s2_std = agg_stats(shell2_vecs, dim)
        parts += [s2_mean, s2_std]

    # RDKit scalar local descriptors
    scalars = atom_local_scalar_features(atom)
    parts.append(scalars)

    x = np.concatenate(parts).astype(np.float32)

    stats = {
        "unknown_center": False,
        "unknown_shell1": unknown1,
        "unknown_shell2": unknown2,
        "deg": int(atom.GetDegree()),
        "radius": radius
    }
    return x, stats


def main():
    df = load_smiles_csv(INPUT_FILE)
    elem_to_vec, dim = load_skipatom_table(SKIPATOM_CSV)

    rows = []
    failed = []
    unknown_center_count = 0
    unknown_shell1_total = 0
    unknown_shell2_total = 0
    total_atoms = 0

    for name, smi in df[["name", "smiles"]].itertuples(index=False):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            failed.append((name, smi, "parse_fail"))
            continue

        if INCLUDE_HYDROGENS:
            try:
                mol = Chem.AddHs(mol)
            except Exception:
                failed.append((name, smi, "addHs_fail"))
                continue

        for atom in mol.GetAtoms():
            total_atoms += 1
            atom_idx = atom.GetIdx()
            sym = atom.GetSymbol()

            x, stats = atom_environment_vector(
                mol, atom_idx, elem_to_vec, dim,
                radius=RADIUS
            )

            if x is None:
                unknown_center_count += 1
                continue

            unknown_shell1_total += stats["unknown_shell1"]
            unknown_shell2_total += stats["unknown_shell2"]

            rows.append({
                "name": name,
                "smiles": smi,
                "atom_index": atom_idx,
                "atom_symbol": sym,
                "degree": stats["deg"],
                "radius": stats["radius"],
                "unknown_shell1": stats["unknown_shell1"],
                "unknown_shell2": stats["unknown_shell2"],
                "vec": x
            })

    print(f"Parsed molecules: {len(df)} | failed molecules: {len(failed)}")
    print(f"Total atoms visited: {total_atoms}")
    print(f"Unknown center atoms skipped: {unknown_center_count}")
    if total_atoms > 0:
        print(f"Unknown neighbour counts (shell1 total): {unknown_shell1_total}")
        print(f"Unknown neighbour counts (shell2 total): {unknown_shell2_total}")

    if not rows:
        raise RuntimeError("No atom environments were featurised. Check SMILES and SkipAtom coverage.")

    X = np.vstack([r["vec"] for r in rows])
    feat_cols = [f"env_{i}" for i in range(X.shape[1])]

    out = pd.DataFrame({
        "name": [r["name"] for r in rows],
        "smiles": [r["smiles"] for r in rows],
        "atom_index": [r["atom_index"] for r in rows],
        "atom_symbol": [r["atom_symbol"] for r in rows],
        "degree": [r["degree"] for r in rows],
        "radius": [r["radius"] for r in rows],
        "unknown_shell1": [r["unknown_shell1"] for r in rows],
        "unknown_shell2": [r["unknown_shell2"] for r in rows],
    })

    out = pd.concat([out, pd.DataFrame(X, columns=feat_cols)], axis=1)
    out.to_csv(OUTPUT_FILE, index=False)
    print("Saved:", OUTPUT_FILE, "shape:", X.shape)

    if failed:
        pd.DataFrame(failed, columns=["name", "smiles", "reason"]).to_csv(
            "skipatom_env_failures.csv", index=False
        )
        print("Saved failures:", "skipatom_env_failures.csv", "n=", len(failed))


if __name__ == "__main__":
    main()



Loaded SkipAtom embeddings for 86 elements. dim=200
Parsed molecules: 111 | failed molecules: 7
Total atoms visited: 1052
Unknown center atoms skipped: 0
Unknown neighbour counts (shell1 total): 0
Unknown neighbour counts (shell2 total): 0
Saved: polymer_skipatom_env_vectors.csv shape: (1052, 1012)
Saved failures: skipatom_env_failures.csv n= 7
